In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
from tqdm import tqdm
from typing import List, Dict
from urllib.parse import urljoin

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
from typing import List, Dict
from urllib.parse import urljoin, urlparse

# --- Конфигурация ---
BASE_URL = "https://lifehacker.ru"
SECTION_PATH = "/topics/technology/"  # актуальный URL рубрики «Технологии»
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
}
DELAY = 1.5
MAX_PAGES = 10

# Пути, которые точно не являются статьями
NON_ARTICLE_PREFIXES = {
    "/topics/", "/category/", "/tag/", "/best/", "/news/", "/life/",
    "/recipes/", "/health/", "/kino/", "/technology/", "/purchases/",
    "/about/", "/reklama/", "/redaktsiya/", "/vakansii/", "/pravila/",
    "/rss", "/telegram", "/vkontakte", "/twitter", "/youtube",
    "/pinterest", "/author/",
}

def get_soup(url: str) -> BeautifulSoup:
    resp = requests.get(url, headers=HEADERS, timeout=10)
    resp.raise_for_status()
    return BeautifulSoup(resp.text, "html.parser")

def is_article_link(href: str) -> bool:
    """Проверяет, похожа ли ссылка на статью (корневой слаг вида /nazvanie-stati/)."""
    if not href or href.startswith("#") or href.startswith("mailto:") or href.startswith("tel:"):
        return False
    parsed = urlparse(href if href.startswith("http") else urljoin(BASE_URL, href))
    if parsed.netloc and parsed.netloc not in ("lifehacker.ru", "www.lifehacker.ru", ""):
        return False  # внешняя ссылка
    path = parsed.path.strip("/")
    if not path:
        return False
    # Статья — это один сегмент пути (например, "gadzhety-za-kotorye-stoit-pereplatit")
    segments = path.split("/")
    if len(segments) != 1:
        return False
    # Проверяем, что не начинается с исключённых префиксов
    for prefix in NON_ARTICLE_PREFIXES:
        if ("/" + path).startswith(prefix):
            return False
    return True

def extract_article_links_from_list_page(soup: BeautifulSoup) -> List[str]:
    links = []
    for a in soup.find_all("a", href=True):
        href = a["href"].split("#")[0].split("?")[0]
        if not is_article_link(href):
            continue
        full_url = urljoin(BASE_URL, href)
        links.append(full_url)
    # убираем дубликаты
    seen = set()
    unique = []
    for l in links:
        if l not in seen:
            seen.add(l)
            unique.append(l)
    return unique

def parse_article_page(url: str) -> Dict[str, str]:
    soup = get_soup(url)

    # Заголовок: h1
    title_tag = soup.find("h1")
    title = title_tag.get_text(strip=True) if title_tag else ""

    # Текст: ищем основной блок контента
    content_div = None
    possible_classes = [
        "entry-content", "article-body", "post-content",
        "content-block", "article__content", "post__content",
        "lh-article__body", "article-content",
    ]
    for cls in possible_classes:
        content_div = soup.find("div", class_=cls)
        if content_div:
            break

    # Запасной вариант: тег <article>
    if not content_div:
        content_div = soup.find("article")

    text_parts = []
    if content_div:
        for p in content_div.find_all("p"):
            txt = p.get_text(strip=True)
            if txt:
                text_parts.append(txt)

    full_text = "\n".join(text_parts)
    return {"title": title, "text": full_text, "url": url}

def main():
    all_article_urls: List[str] = []

    for page_num in range(1, MAX_PAGES + 1):
        # Пагинация: первая страница без параметра, остальные — ?page=N
        if page_num == 1:
            list_url = f"{BASE_URL}{SECTION_PATH}"
        else:
            list_url = f"{BASE_URL}{SECTION_PATH}?page={page_num}"

        print(f"Страница {page_num}: {list_url}")
        try:
            soup = get_soup(list_url)
            links = extract_article_links_from_list_page(soup)
            print(f"  Найдено ссылок: {len(links)}")
            all_article_urls.extend(links)
        except Exception as e:
            print(f"  Ошибка: {e}")
        time.sleep(DELAY)

    # убираем дубликаты между страницами
    seen = set()
    unique_urls = []
    for u in all_article_urls:
        if u not in seen:
            seen.add(u)
            unique_urls.append(u)
    all_article_urls = unique_urls
    print(f"\nВсего уникальных ссылок на статьи: {len(all_article_urls)}")

    # --- Перебираем ссылки и парсим каждую статью ---
    articles_data: List[Dict] = []
    for idx, url in enumerate(all_article_urls):
        print(f"[{idx+1}/{len(all_article_urls)}] {url}")
        try:
            data = parse_article_page(url)
            articles_data.append(data)
        except Exception as e:
            print(f"  Ошибка: {e}")
        time.sleep(DELAY)

    # --- Создаём DataFrame ---
    if articles_data:
        df = pd.DataFrame(articles_data)
        # убираем строки без заголовка
        df = df[df["title"].str.len() > 0].reset_index(drop=True)
    else:
        df = pd.DataFrame(columns=["title", "text", "url"])

    print(f"\nДатафрейм: {df.shape}")
    print(df.head(10))
    return df

if __name__ == "__main__":
    df_result = main()


Страница 1: https://lifehacker.ru/topics/technology/
  Найдено ссылок: 46
Страница 2: https://lifehacker.ru/topics/technology/?page=2
  Найдено ссылок: 47
Страница 3: https://lifehacker.ru/topics/technology/?page=3
  Найдено ссылок: 47
Страница 4: https://lifehacker.ru/topics/technology/?page=4
  Найдено ссылок: 46
Страница 5: https://lifehacker.ru/topics/technology/?page=5
  Найдено ссылок: 47
Страница 6: https://lifehacker.ru/topics/technology/?page=6
  Найдено ссылок: 47
Страница 7: https://lifehacker.ru/topics/technology/?page=7
  Найдено ссылок: 47
Страница 8: https://lifehacker.ru/topics/technology/?page=8
  Найдено ссылок: 47
Страница 9: https://lifehacker.ru/topics/technology/?page=9
  Найдено ссылок: 46
Страница 10: https://lifehacker.ru/topics/technology/?page=10
  Найдено ссылок: 47

Всего уникальных ссылок на статьи: 314
[1/314] https://lifehacker.ru/recipes/
[2/314] https://lifehacker.ru/health/
[3/314] https://lifehacker.ru/topics/
[4/314] https://lifehacker.ru/gadzhety-z